# 04 — vLLM: Needle-in-a-Haystack Baseline

This notebook runs the same NIAH benchmark as notebook 02, but using
[vLLM](https://github.com/vllm-project/vllm) with Qwen3-8B **without
KV cache compression**.

vLLM uses PagedAttention for efficient memory management. This serves
as the quality and throughput baseline for comparison against
KeyDiffPress in notebook 05.

Results are saved to `results/vllm/` for comparison in notebook 05.

## 1 Configuration

In [1]:
MODEL_NAME = "Qwen/Qwen3-8B"

COMPRESSION_RATIOS = [0.01, 0.25, 0.50, 0.75]

NEEDLE_DEPTHS = [0, 25, 50, 75, 100]

MAX_CONTEXT_LENGTHS = [4096, 8192]

MAX_NEW_TOKENS = 64

PRESS_CONFIGS = {
    "full_replacement": lambda cr: 
    {
        "model": MODEL_NAME,
        "dtype": "auto",
        "gpu_memory_utilization": 0.90,
        "max_model_len": max(MAX_CONTEXT_LENGTHS) + MAX_NEW_TOKENS + 256,
        "trust_remote_code": True,
        "attention_config": {"backend": "FLASH_ATTN"},
        "kv_compression_algorithm": "full_replacement",
        "kv_compression_ratio": cr,
        "enable_prefix_caching": False
    },
    "filtering": lambda cr: {
        "model": MODEL_NAME,
        "dtype": "auto",
        "gpu_memory_utilization": 0.90,
        "max_model_len": max(MAX_CONTEXT_LENGTHS) + MAX_NEW_TOKENS + 256,
        "trust_remote_code": True,
        "attention_config": {"backend": "FLASH_ATTN"},
        "kv_compression_algorithm": "filtering",
        "kv_compression_ratio": cr,
    },
}

In [2]:
import sys
import os

FORK_DIR = "/opt/app-root/src/vllm-fork"

if os.path.isdir(FORK_DIR) and os.listdir(FORK_DIR):
    sys.path.insert(0, FORK_DIR)
    import vllm
    print(f"Using FORK vLLM (version: {vllm.__version__})")
else:
    import vllm
    print(f"Using SYSTEM vLLM (version: {vllm.__version__})")

/opt/app-root/src/vllm-fork/vllm/__init__.py:7: RuntimeWarning: Failed to read commit hash:
No module named 'vllm._version'
  from .version import __version__, __version_tuple__  # isort:skip


Using FORK vLLM (version: dev)


In [3]:
import gc
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected.")

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
allocated_gb = torch.cuda.memory_allocated() / 1e9
reserved_gb = torch.cuda.memory_reserved() / 1e9

print(f"GPU:        {torch.cuda.get_device_name(0)}")
print(f"VRAM:       {vram_gb:.1f} GB total")
print(f"Allocated:  {allocated_gb:.2f} GB")
print(f"Reserved:   {reserved_gb:.2f} GB")
print(f"Free:       {vram_gb - reserved_gb:.1f} GB (approx)")

if allocated_gb > 1.0:
    print(
        "\n⚠  GPU memory is not free — a model from another notebook may still be loaded.\n"
        "   Restart this kernel before proceeding."
    )

def cleanup_vllm(llm):
    llm.llm_engine.engine_core.shutdown()
    del llm
    gc.collect()
    torch.cuda.empty_cache()    

GPU:        NVIDIA A100-SXM4-40GB
VRAM:       42.4 GB total
Allocated:  0.00 GB
Reserved:   0.00 GB
Free:       42.4 GB (approx)


## 2. Load Haystack Dataset

Paul Graham essays used as haystack filler, following the original
Needle-in-a-Haystack benchmark by [Kamradt (2023)](https://github.com/gkamradt/LLMTest_NeedleInAHaystack).

In [4]:
from datasets import load_dataset

haystack_ds = load_dataset("alessiodevoto/paul_graham_essays", split="test")
haystack_df = haystack_ds.to_pandas()

print(f"Haystack dataset loaded: {len(haystack_df)} rows")
print(f"Columns: {list(haystack_df.columns)}")
print(f"Needle: {haystack_df['needle'].iloc[0][:100]}...")
print(f"Question: {haystack_df['question'].iloc[0]}")

Haystack dataset loaded: 1 rows
Columns: ['context', 'needle', 'question', 'answer_prefix', 'max_new_tokens']
Needle: 

Remember, the best thing to do in San Francisco is eat a sandwich and sit in Dolores Park on a sun...
Question: 

Question: Based on the content of the book, what is the best thing to do in San Francisco?


## 3. Prepare Prompts

For each (max_context_length, needle_depth) combination, we build the
haystack with needle inserted, then apply the model's chat template
to produce the final prompt for vLLM.

In [5]:
from transformers import AutoTokenizer
from eval_utils import insert_needle_in_haystack

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

prompt_configs = []

for max_ctx_len in MAX_CONTEXT_LENGTHS:
    niah_df = insert_needle_in_haystack(
        haystack_df, tokenizer, max_ctx_len, NEEDLE_DEPTHS,
    )

    for idx, row in niah_df.iterrows():
        user_msg = row["context"] + "\n\n" + row["question"]
        messages = [{"role": "user", "content": user_msg}]
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )
        if row["answer_prefix"]:
            prompt += row["answer_prefix"]

        prompt_configs.append({
            "prompt": prompt,
            "max_context_length": max_ctx_len,
            "needle_depth": row["needle_depth"],
            "needle": row["needle"],
            "question": row["question"],
        })

print(f"Prepared {len(prompt_configs)} prompts")

Token indices sequence length is longer than the specified maximum sequence length for this model (699555 > 131072). Running this sequence through the model will result in indexing errors


Prepared 10 prompts


## 4. Run Batch Inference

vLLM processes all prompts in a single batch via its internal scheduler.

In [ ]:
import time
import json
from eval_utils import calculate_niah_metrics, rouge_l_f_scores
from vllm import LLM, SamplingParams

sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=MAX_NEW_TOKENS,
)

def get_gpu_memory_used_gb() -> float:
    """Actual GPU memory used, measured at the CUDA driver level.
    Works for vLLM which manages its own memory pool outside PyTorch."""
    free, total = torch.cuda.mem_get_info()
    return (total - free) / 1e9

prompts = [pc["prompt"] for pc in prompt_configs]

all_results = []
for press_name, press_factory in PRESS_CONFIGS.items():
    for ratio in COMPRESSION_RATIOS:
        llm = None
        try:
            press = press_factory(ratio)
            llm = LLM(**press)
    
            mem_before = get_gpu_memory_used_gb()
            start = time.perf_counter()
    
            outputs = llm.generate(prompts, sampling_params)
    
            batch_elapsed = time.perf_counter() - start
            mem_after = get_gpu_memory_used_gb()
            peak_mem = max(mem_before, mem_after)
    
            for i, output in enumerate(outputs):
                predicted_answer = output.outputs[0].text.strip()
                pc = prompt_configs[i]
    
                result = {
                    "framework": "vllm",
                    "press": press_name,
                    "compression_ratio": ratio,
                    "max_context_length": pc["max_context_length"],
                    "needle_depth": pc["needle_depth"],
                    "needle": pc["needle"],
                    "question": pc["question"],
                    "predicted_answer": predicted_answer,
                    "elapsed_sec": round(batch_elapsed / len(prompts), 3),
                    "peak_gpu_mem_gb": round(peak_mem, 3),
                    "task": "needle_in_haystack",
                }
                all_results.append(result)
    
            total_gen_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
            throughput = total_gen_tokens / batch_elapsed if batch_elapsed > 0 else 0
    
            print(f"Batch done in {batch_elapsed:.1f}s — {throughput:.1f} tok/s — peak mem={peak_mem:.2f} GB")
        finally:
            if llm is not None:
                cleanup_vllm(llm)
print(f"Total results: {len(all_results)}")

INFO 11-04 07:13:42 [utils.py:233] non-default args: {'trust_remote_code': True, 'max_model_len': 8512, 'enable_prefix_caching': False, 'disable_log_stats': True, 'attention_config': AttentionConfig(backend=<AttentionBackendEnum.FLASH_ATTN: 'vllm.v1.attention.backends.flash_attn.FlashAttentionBackend'>, flash_attn_version=None, use_prefill_decode_attention=False, flash_attn_max_num_splits_for_cuda_graph=32, use_cudnn_prefill=False, use_trtllm_ragged_deepseek_prefill=False, use_trtllm_attention=None, disable_flashinfer_prefill=True, disable_flashinfer_q_quantization=False, use_prefill_query_quantization=False), 'kv_compression_algorithm': 'full_replacement', 'kv_compression_ratio': 0.01, 'model': 'Qwen/Qwen3-8B'}
INFO 11-04 07:13:43 [model.py:533] Resolved architecture: Qwen3ForCausalLM
INFO 11-04 07:13:43 [model.py:1582] Using max model len 8512
INFO 11-04 07:13:43 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 11-04 07:13:43 [vllm.py:795] Asynchro

/opt/app-root/src/vllm-fork/vllm/__init__.py:7: RuntimeWarning: Failed to read commit hash:
No module named 'vllm._version'
  from .version import __version__, __version_tuple__  # isort:skip


(EngineCore pid=2673) INFO 11-04 07:13:54 [core.py:103] Initializing a V1 LLM engine (vdev) with config: model='Qwen/Qwen3-8B', speculative_config=None, tokenizer='Qwen/Qwen3-8B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=8512, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collect_detailed_traces=None, kv_cach

(EngineCore pid=2673) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(EngineCore pid=2673) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
Loading safetensors checkpoint shards:   0% Completed | 0/5 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  20% Completed | 1/5 [00:29<01:58, 29.56s/it]
Loading safetensors checkpoint shards:  40% Completed | 2/5 [01:03<01:36, 32.23s/it]
Loading safetensors checkpoint shards:  60% Completed | 3/5 [01:33<01:02, 31.28s/it]
Loading safetensors checkpoint shards:  80% Completed | 4/5 [01:58<00:28, 28.52s/it]
Loading safetensors checkpoint shards: 100% Completed | 5/5 [02:07<00:00, 21.65s/it]
Loading safetensors checkpoint shards: 100% Completed |

(EngineCore pid=2673) INFO 11-04 07:16:05 [default_loader.py:384] Loading weights took 127.62 seconds
(EngineCore pid=2673) INFO 11-04 07:16:06 [gpu_model_runner.py:4627] Model loading took 15.27 GiB memory and 128.887569 seconds
(EngineCore pid=2673) INFO 11-04 07:16:16 [backends.py:988] Using cache directory: /opt/app-root/src/.cache/vllm/torch_compile_cache/25e509be41/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=2673) INFO 11-04 07:16:16 [backends.py:1048] Dynamo bytecode transform time: 9.31 s
(EngineCore pid=2673) INFO 11-04 07:16:22 [backends.py:371] Cache the graph of compile range (1, 8192) for later use
(EngineCore pid=2673) INFO 11-04 07:16:27 [backends.py:387] Compiling a graph for compile range (1, 8192) takes 11.29 s
(EngineCore pid=2673) INFO 11-04 07:16:30 [decorators.py:627] saved AOT compiled function to /opt/app-root/src/.cache/vllm/torch_compile_cache/torch_aot_compile/5ef489379059e41f5749b214d288ae8e189b8d3978eb980ff416b4c1329ddfe9/rank_0_0/model
(Engi

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:02<00:00, 21.34it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:01<00:00, 25.86it/s]


... (43 outputs left out)

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:02<00:00, 20.83it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:01<00:00, 25.72it/s]


(EngineCore pid=3782) INFO 11-04 07:28:42 [gpu_model_runner.py:5807] Graph capturing finished in 5 secs, took 0.52 GiB
(EngineCore pid=3782) INFO 11-04 07:28:42 [gpu_worker.py:617] CUDA graph pool memory: 0.52 GiB (actual), 0.52 GiB (estimated), difference: 0.0 GiB (0.8%).
(EngineCore pid=3782) INFO 11-04 07:28:42 [core.py:281] init engine (profile, create kv cache, warmup model) took 14.21 seconds
(EngineCore pid=3782) INFO 11-04 07:28:43 [vllm.py:795] Asynchronous scheduling is enabled.
INFO 11-04 07:28:43 [llm.py:391] Supported tasks: ['generate']


Rendering prompts:   0%|          | 0/10 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

## 5. Score Predictions

In [ ]:
import pandas as pd

df = pd.DataFrame(all_results)

metrics = calculate_niah_metrics(df)
df["rouge_l_f"] = rouge_l_f_scores(metrics)

summary = (
    df.groupby(["press", "compression_ratio", "max_context_length", "needle_depth"])
    .agg(
        rouge_l=("rouge_l_f", "mean"),
        mean_time=("elapsed_sec", "mean"),
        peak_mem=("peak_gpu_mem_gb", "mean"),
    )
    .round(4)
)

print(summary.to_string())

## 6. Save Results

In [ ]:
import os

os.makedirs("results/vllm", exist_ok=True)

predictions_path = "results/vllm/predictions.csv"
df.to_csv(predictions_path, index=False)
print(f"Saved predictions to {predictions_path}")

metrics_path = "results/vllm/metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Saved metrics to {metrics_path}")